In [1]:
import polars as pl
import math
from pathlib import Path

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(250)

polars.config.Config

In [2]:
GTFS = Path("raw/processed_gtfs")

routes = pl.read_parquet(GTFS / "routes.parquet")
trips = pl.read_parquet(GTFS / "trips.parquet")
stop_times = pl.read_parquet(GTFS / "stop_times.parquet")
stops = pl.read_parquet(GTFS / "stops.parquet")
shapes = pl.read_parquet(GTFS / "shapes.parquet")

In [3]:
# ============================================================
# Clean stop coordinates
# ============================================================

stops = stops.with_columns(
    pl.col("stop_lat")
        .str.strip_chars()
        .cast(pl.Float64),

    pl.col("stop_lon")
        .str.strip_chars()
        .cast(pl.Float64)
)

In [4]:
# ============================================================
# GTFS time parser
# Handles times like 25:13:00
# ============================================================

def gtfs_to_seconds(t):

    if t is None:
        return None

    h, m, s = map(int, t.split(":"))

    return h * 3600 + m * 60 + s

In [5]:
# ============================================================
# Convert schedule times
# ============================================================

stop_times = stop_times.with_columns(

    pl.col("arrival_time")
    .map_elements(
        gtfs_to_seconds,
        return_dtype=pl.Int64
    )
    .alias("arrival_seconds"),

    pl.col("departure_time")
    .map_elements(
        gtfs_to_seconds,
        return_dtype=pl.Int64
    )
    .alias("departure_seconds")

)

In [6]:
# ============================================================
# Sort tables
# ============================================================

stop_times = stop_times.sort(
    ["trip_id", "stop_sequence"]
)

shapes = shapes.sort(
    ["shape_id", "shape_pt_sequence"]
)

In [7]:
# ============================================================
# Previous shape point
# ============================================================

shapes = shapes.with_columns(

    pl.col("shape_pt_lat")
    .shift()
    .over("shape_id")
    .alias("prev_lat"),

    pl.col("shape_pt_lon")
    .shift()
    .over("shape_id")
    .alias("prev_lon")

)

In [8]:
# ============================================================
# Compute edge length (Pure Polars)
# ============================================================

R = 6371000.0

lat1 = pl.col("prev_lat") * math.pi / 180
lon1 = pl.col("prev_lon") * math.pi / 180

lat2 = pl.col("shape_pt_lat") * math.pi / 180
lon2 = pl.col("shape_pt_lon") * math.pi / 180

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    (dlat / 2).sin().pow(2)
    +
    lat1.cos()
    *
    lat2.cos()
    *
    (dlon / 2).sin().pow(2)
)

c = 2 * pl.arctan2(
    a.sqrt(),
    (1 - a).sqrt()
)

shapes = shapes.with_columns(

    pl.when(
        pl.col("prev_lat").is_null()
    )

    .then(0.0)

    .otherwise(
        R * c
    )

    .alias("edge_length")

)

In [9]:
# ============================================================
# Cumulative distance along every shape
# ============================================================

shapes = shapes.with_columns(

    pl.col("edge_length")
      .cum_sum()
      .over("shape_id")
      .alias("shape_distance")

)

In [10]:
# ============================================================
# Create stop -> shape lookup
# ============================================================

stop_shape = (

    stop_times

    .join(

        trips.select([
            "trip_id",
            "shape_id"
        ]),

        on="trip_id"

    )

    .join(

        stops.select([
            "stop_id",
            "stop_lat",
            "stop_lon"
        ]),

        on="stop_id"

    )

)

In [11]:
# ============================================================
# Join all shape points
# ============================================================

stop_shape = stop_shape.join(

    shapes.select([
        "shape_id",
        "shape_pt_sequence",
        "shape_pt_lat",
        "shape_pt_lon",
        "shape_distance"
    ]),

    on="shape_id"

)

In [12]:
# ============================================================
# Distance stop -> shape point
# Pure Polars
# ============================================================

lat1 = pl.col("stop_lat") * math.pi / 180
lon1 = pl.col("stop_lon") * math.pi / 180

lat2 = pl.col("shape_pt_lat") * math.pi / 180
lon2 = pl.col("shape_pt_lon") * math.pi / 180

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    (dlat / 2).sin().pow(2)
    +
    lat1.cos()
    *
    lat2.cos()
    *
    (dlon / 2).sin().pow(2)
)

c = 2 * pl.arctan2(
    a.sqrt(),
    (1 - a).sqrt()
)

stop_shape = stop_shape.with_columns(

    (R * c)

    .alias("distance_to_shape")

)

In [ ]:
# ============================================================
# Keep nearest shape point
# ============================================================

stop_shape_mapping = (

    stop_shape

    .sort("distance_to_shape")

    .group_by([
        "trip_id",
        "stop_id"
    ])

    .first()

    .select([

        "trip_id",

        "shape_id",

        "stop_id",

        "shape_pt_sequence",

        "shape_distance"

    ])

)

In [ ]:
print(stop_shape_mapping.shape)

stop_shape_mapping.head()

(220373, 5)


trip_id,shape_id,stop_id,shape_pt_sequence,shape_distance
str,str,i64,i64,f64
"""UP_C6-Saturday-046300_B6_219""","""B60099""",300654,330008,12962.736779
"""MV_C6-Weekday-059200_M4_428""","""M040918""",400611,520002,10692.266661
"""FB_C6-Sunday-096600_B41_222""","""B410154""",308987,180004,10974.383551
"""FB_C6-Weekday-SDon-123400_B41_…","""B410153""",308348,200010,5736.850764
"""FB_C6-Weekday-SDon-047900_B41_…","""B410151""",308987,430004,11601.913411


In [ ]:
(
    stop_shape_mapping
    .filter(pl.col("trip_id") == "FB_C6-Saturday-066200_B41_230")
    .sort("shape_distance")
    .select([
        "stop_id",
        "shape_distance"
    ])
)

stop_id,shape_distance
i64,f64
307403,0.0
308051,632.461173
308845,1026.065569
306901,1209.156087
308884,1650.611022
303295,2139.612333
308780,2397.052739
303297,2610.013074
303300,3006.46652


In [ ]:
(
    stop_shape_mapping
    .filter(pl.col("trip_id") == "FB_C6-Saturday-066200_B41_230")
    .sort("shape_distance")
    .with_columns(
        (
            pl.col("shape_distance")
            - pl.col("shape_distance").shift(1)
        ).alias("segment_length")
    )
)

trip_id,shape_id,stop_id,shape_pt_sequence,shape_distance,segment_length
str,str,i64,i64,f64,f64
"""FB_C6-Saturday-066200_B41_230""","""B410126""",307403,10001,0.0,null
"""FB_C6-Saturday-066200_B41_230""","""B410126""",308051,10014,632.461173,632.461173
"""FB_C6-Saturday-066200_B41_230""","""B410126""",308845,20012,1026.065569,393.604396
"""FB_C6-Saturday-066200_B41_230""","""B410126""",306901,30005,1209.156087,183.090518
"""FB_C6-Saturday-066200_B41_230""","""B410126""",308884,40006,1650.611022,441.454935
"""FB_C6-Saturday-066200_B41_230""","""B410126""",303295,50009,2139.612333,489.001311
"""FB_C6-Saturday-066200_B41_230""","""B410126""",308780,60005,2397.052739,257.440406
"""FB_C6-Saturday-066200_B41_230""","""B410126""",303297,70004,2610.013074,212.960335
"""FB_C6-Saturday-066200_B41_230""","""B410126""",303300,80007,3006.46652,396.453445


In [ ]:
stop_shape_mapping.write_parquet("processed/stop_shape_mapping.parquet")

In [ ]:
segments = (
    stop_times
    .with_columns([

        pl.col("stop_id")
        .shift(-1)
        .over("trip_id")
        .alias("next_stop_id"),

        pl.col("stop_sequence")
        .shift(-1)
        .over("trip_id")
        .alias("next_stop_sequence"),

        pl.col("arrival_seconds")
        .shift(-1)
        .over("trip_id")
        .alias("next_arrival"),

        pl.col("departure_seconds")
        .shift(-1)
        .over("trip_id")
        .alias("next_departure")

    ])
    .filter(
        pl.col("next_stop_id").is_not_null()
    )
)

In [ ]:
segments = segments.with_columns(

    (
        pl.col("next_arrival")
        -
        pl.col("departure_seconds")
    )

    .alias("scheduled_travel_time")

)

In [ ]:
segments = segments.join(

    trips.select([
        "trip_id",
        "route_id",
        "direction_id",
        "shape_id"
    ]),

    on="trip_id",
    how="left"

)

In [ ]:
segments = segments.join(

    stops.select([
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    ]),

    on="stop_id",
    how="left"

)

In [ ]:
segments = segments.rename({

    "stop_name": "start_stop_name",
    "stop_lat": "start_lat",
    "stop_lon": "start_lon"

})

In [ ]:
segments = segments.join(

    stops.select([
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    ]),

    left_on="next_stop_id",
    right_on="stop_id",
    how="left"

)

In [ ]:
segments = segments.rename({

    "stop_name": "end_stop_name",
    "stop_lat": "end_lat",
    "stop_lon": "end_lon"

})

In [ ]:
segments = (
    segments
    .join(
        stop_shape_mapping.select([
            "trip_id",
            "stop_id",
            "shape_distance"
        ]),
        on=["trip_id", "stop_id"],
        how="left"
    )
)

In [ ]:
segments = (
    segments
    .join(
        stop_shape_mapping.select([
            "trip_id",
            "stop_id",
            "shape_distance"
        ]).rename({
            "stop_id": "next_stop_id",
            "shape_distance": "next_shape_distance"
        }),
        on=["trip_id", "next_stop_id"],
        how="left"
    )
)

In [ ]:
segments = segments.with_columns(

    (
        pl.col("next_shape_distance")
        -
        pl.col("shape_distance")
    )
    .abs()
    .alias("segment_length")

)

In [ ]:
segments = segments.with_columns(

    pl.concat_str([

        pl.col("route_id"),

        pl.col("direction_id").cast(pl.String),

        pl.col("shape_id"),

        pl.col("stop_id").cast(pl.String),

        pl.col("next_stop_id").cast(pl.String)

    ], separator="_").alias("segment_id")

)

In [ ]:
segment_network = (

    segments

    .select([

        "segment_id",

        "route_id",
        "direction_id",
        "shape_id",

        "stop_id",
        "next_stop_id",

        "stop_sequence",
        "next_stop_sequence",

        "start_stop_name",
        "end_stop_name",

        "start_lat",
        "start_lon",

        "end_lat",
        "end_lon",

        "shape_distance",
        "next_shape_distance",

        "segment_length",

        "scheduled_travel_time"

    ])

    .unique(subset=["segment_id"])

    .sort([
        "route_id",
        "direction_id",
        "shape_distance"
    ])

)

In [ ]:
segment_schedule = (

    segments

    .select([

        "trip_id",

        "route_id",
        "direction_id",

        "segment_id",

        "departure_seconds",

        "next_arrival",

        "scheduled_travel_time"

    ])

    .sort([
        "trip_id",
        "departure_seconds"
    ])

)

In [ ]:
OUTPUT = Path("processed")

OUTPUT.mkdir(exist_ok=True)

segment_network.write_parquet(
    OUTPUT / "segment_network.parquet"
)

segment_schedule.write_parquet(
    OUTPUT / "segment_schedule.parquet"
)

In [ ]:
segment_network.head()

segment_id,route_id,direction_id,shape_id,stop_id,next_stop_id,stop_sequence,next_stop_sequence,start_stop_name,end_stop_name,start_lat,start_lon,end_lat,end_lon,shape_distance,next_shape_distance,segment_length,scheduled_travel_time
str,str,i64,str,i64,i64,i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,i64
"""B41_0_B410152_308381_303269""","""B41""",0,"""B410152""",308381,303269,1,2,"""E 70 ST/VETERANS AV""","""VETERANS AV/AVENUE T""",40.619935,-73.908708,40.62002,-73.910961,0.0,194.519241,194.519241,61
"""B41_0_B410165_308381_303269""","""B41""",0,"""B410165""",308381,303269,1,2,"""E 70 ST/VETERANS AV""","""VETERANS AV/AVENUE T""",40.619935,-73.908708,40.62002,-73.910961,0.0,194.519241,194.519241,55
"""B41_0_B410135_303215_300163""","""B41""",0,"""B410135""",303215,300163,1,2,"""FLATBUSH AV/KINGS PLAZA""","""FLATBUSH AV/AVENUE T""",40.609251,-73.92154,40.611476,-73.92407,0.0,326.685685,326.685685,92
"""B41_0_B410157_308381_303269""","""B41""",0,"""B410157""",308381,303269,1,2,"""E 70 ST/VETERANS AV""","""VETERANS AV/AVENUE T""",40.619935,-73.908708,40.62002,-73.910961,0.0,194.519241,194.519241,44
"""B41_0_B410151_308381_303269""","""B41""",0,"""B410151""",308381,303269,1,2,"""E 70 ST/VETERANS AV""","""VETERANS AV/AVENUE T""",40.619935,-73.908708,40.62002,-73.910961,0.0,194.519241,194.519241,39


In [ ]:
segment_schedule.head()

trip_id,route_id,direction_id,segment_id,departure_seconds,next_arrival,scheduled_travel_time
str,str,i64,str,i64,i64,i64
"""FB_C6-Saturday-003500_B41_201""","""B41""",0,"""B41_0_B410153_303215_300163""",2100,2166,66
"""FB_C6-Saturday-003500_B41_201""","""B41""",0,"""B41_0_B410153_300163_303218""",2166,2232,66
"""FB_C6-Saturday-003500_B41_201""","""B41""",0,"""B41_0_B410153_303218_303219""",2232,2281,49
"""FB_C6-Saturday-003500_B41_201""","""B41""",0,"""B41_0_B410153_303219_306273""",2281,2330,49
"""FB_C6-Saturday-003500_B41_201""","""B41""",0,"""B41_0_B410153_306273_303222""",2330,2400,70


In [ ]:
shape_points = (
    shapes
    .select([
        "shape_id",
        "shape_pt_sequence",
        "shape_pt_lat",
        "shape_pt_lon",
        "shape_distance"
    ])
    .sort([
        "shape_id",
        "shape_pt_sequence"
    ])
)

shape_points.write_parquet(
    "processed/shape_points.parquet"
)

In [ ]:
trip_shapes = (
    trips
    .select([
        "trip_id",
        "route_id",
        "direction_id",
        "shape_id"
    ])
)

trip_shapes.write_parquet(
    "processed/trip_shapes.parquet"
)

In [ ]:
vehicle = pl.read_parquet("raw/vehicle_positions/2026-04-06.parquet")

(
    vehicle
    .filter(pl.col("trip_id") == "MV_A6-Sunday-114600_M4_454")
    .sort("timestamp")
    .select([
        "timestamp",
        "vehicle_id",
        "trip_id",
        "stop_id",
        "latitude",
        "longitude"
    ])
    .head(100)
)

timestamp,vehicle_id,trip_id,stop_id,latitude,longitude
u64,str,str,str,f32,f32
1775433585,"""MTA NYCT_9777""","""MV_A6-Sunday-114600_M4_454""","""400614""",40.827335,-73.949661
1775433613,"""MTA NYCT_9777""","""MV_A6-Sunday-114600_M4_454""","""400614""",40.827431,-73.949593
1775433641,"""MTA NYCT_9777""","""MV_A6-Sunday-114600_M4_454""","""400615""",40.827991,-73.949181
1775433668,"""MTA NYCT_9777""","""MV_A6-Sunday-114600_M4_454""","""400615""",40.828377,-73.948898
1775433695,"""MTA NYCT_9777""","""MV_A6-Sunday-114600_M4_454""","""400615""",40.828835,-73.94857
1775433751,"""MTA NYCT_9777""","""MV_A6-Sunday-114600_M4_454""","""400616""",40.829315,-73.948219
1775433751,"""MTA NYCT_9777""","""MV_A6-Sunday-114600_M4_454""","""400616""",40.829971,-73.947739
1775433779,"""MTA NYCT_9777""","""MV_A6-Sunday-114600_M4_454""","""400616""",40.82988,-73.947807
1775433807,"""MTA NYCT_9777""","""MV_A6-Sunday-114600_M4_454""","""400617""",40.830727,-73.947189


In [ ]:
import polars as pl
import math
from pathlib import Path

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(250)

polars.config.Config

In [ ]:
import polars as pl
vehicle_positions = pl.read_parquet("raw/vehicle_positions/2026-04-06.parquet")
trip_updates = pl.read_parquet("raw/trip_updates/2026-04-06.parquet")

vehicle_positions.join(
        trip_updates.select([
            "trip_id",
            "stop_id",
            "stop_sequence",
            "arrival_time",
            "departure_time"
        ]),
        on=["trip_id", "stop_id"],
        how="left"
    )

trip_id,route_id,vehicle_id,direction_id,start_time,start_date,vehicle_label,latitude,longitude,bearing,stop_id,timestamp,stop_sequence,arrival_time,departure_time
str,str,str,u32,str,str,str,f32,f32,f32,str,u64,u32,i64,i64
"""MV_A6-Sunday-114600_M4_454""","""M4""","""MTA NYCT_9777""",0,"""""","""20260405""","""""",40.827335,-73.949661,53.318657,"""400614""",1775433585,null,null,null
"""UP_A6-Sunday-115800_B36_421""","""B6""","""MTA NYCT_7174""",1,"""""","""20260405""","""""",40.632603,-73.931915,183.54184,"""300627""",1775433590,null,null,null
"""MV_A6-Sunday-117800_M4_439""","""M4""","""MTA NYCT_9715""",0,"""""","""20260405""","""""",40.777325,-73.961388,54.293308,"""400034""",1775433578,null,null,null
"""UP_A6-Sunday-115500_B36_413""","""B6""","""MTA NYCT_4872""",0,"""""","""20260405""","""""",40.632946,-73.918694,276.660065,"""300560""",1775433590,null,null,null
"""UP_A6-Sunday-114600_B6_231""","""B6""","""MTA NYCT_4865""",0,"""""","""20260405""","""""",40.632458,-73.917992,32.660912,"""300560""",1775433575,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""44409259-CPPA6-CP_A6-Weekday-0…","""Q25""","""MTABC_594""",1,"""""","""20260406""","""""",40.722046,-73.809738,291.401733,"""550989""",1775519959,null,null,null
"""44409755-CPPA6-CP_A6-Weekday-0…","""Q25""","""MTABC_191""",0,"""""","""20260406""","""""",40.699886,-73.804672,20.786932,"""701020""",1775519960,null,null,null
"""44409801-CPPA6-CP_A6-Weekday-0…","""Q25""","""MTABC_187""",0,"""""","""20260406""","""""",40.770271,-73.832947,111.996048,"""552832""",1775519934,null,null,null


In [ ]:
import polars as pl

trip = "MV_A6-Sunday-114600_M4_454"

vp = (
    pl.read_parquet("raw/vehicle_positions/2026-04-06.parquet")
    .filter(pl.col("trip_id") == trip)
    .sort("timestamp")
)

tu = (
    pl.read_parquet("raw/trip_updates/2026-04-06.parquet")
    .filter(pl.col("trip_id") == trip)
    .sort("stop_sequence")
)

print(vp.select([
    "timestamp",
    "stop_id"
]))

print()

print(tu.select([
    "stop_sequence",
    "stop_id",
    "arrival_time",
    "departure_time"
]))

shape: (62, 2)
┌────────────┬─────────┐
│ timestamp  ┆ stop_id │
│ ---        ┆ ---     │
│ u64        ┆ str     │
╞════════════╪═════════╡
│ 1775433585 ┆ 400614  │
│ 1775433613 ┆ 400614  │
│ 1775433641 ┆ 400615  │
│ 1775433668 ┆ 400615  │
│ 1775433695 ┆ 400615  │
│ …          ┆ …       │
│ 1775435357 ┆ 400636  │
│ 1775435384 ┆ 400636  │
│ 1775435411 ┆ 400636  │
│ 1775435439 ┆ 400636  │
│ 1775435467 ┆ 400636  │
└────────────┴─────────┘

shape: (618, 4)
┌───────────────┬─────────┬──────────────┬────────────────┐
│ stop_sequence ┆ stop_id ┆ arrival_time ┆ departure_time │
│ ---           ┆ ---     ┆ ---          ┆ ---            │
│ u32           ┆ str     ┆ i64          ┆ i64            │
╞═══════════════╪═════════╪══════════════╪════════════════╡
│ 55            ┆ 400614  ┆ 1775433645   ┆ 1775433645     │
│ 55            ┆ 400614  ┆ 1775433665   ┆ 1775433665     │
│ 55            ┆ 400614  ┆ 1775433646   ┆ 1775433646     │
│ 56            ┆ 400615  ┆ 1775433703   ┆ 1775433703     │
│ 5

In [ ]:
joined = (
    vp.join(
        tu.select([
            "trip_id",
            "stop_id",
            "stop_sequence",
            "arrival_time",
            "departure_time"
        ]),
        on=["trip_id", "stop_id"],
        how="left"
    )
)

print(
    joined.select([
        "timestamp",
        "stop_id",
        "stop_sequence",
        "arrival_time",
        "departure_time"
    ])
)

shape: (1_122, 5)
┌────────────┬─────────┬───────────────┬──────────────┬────────────────┐
│ timestamp  ┆ stop_id ┆ stop_sequence ┆ arrival_time ┆ departure_time │
│ ---        ┆ ---     ┆ ---           ┆ ---          ┆ ---            │
│ u64        ┆ str     ┆ u32           ┆ i64          ┆ i64            │
╞════════════╪═════════╪═══════════════╪══════════════╪════════════════╡
│ 1775433585 ┆ 400614  ┆ 55            ┆ 1775433645   ┆ 1775433645     │
│ 1775433585 ┆ 400614  ┆ 55            ┆ 1775433665   ┆ 1775433665     │
│ 1775433585 ┆ 400614  ┆ 55            ┆ 1775433646   ┆ 1775433646     │
│ 1775433613 ┆ 400614  ┆ 55            ┆ 1775433645   ┆ 1775433645     │
│ 1775433613 ┆ 400614  ┆ 55            ┆ 1775433665   ┆ 1775433665     │
│ …          ┆ …       ┆ …             ┆ …            ┆ …              │
│ 1775435357 ┆ 400636  ┆ null          ┆ null         ┆ null           │
│ 1775435384 ┆ 400636  ┆ null          ┆ null         ┆ null           │
│ 1775435411 ┆ 400636  ┆ null    

In [ ]:
validation = (
    joined.with_columns(
        (
            pl.col("timestamp") - pl.col("arrival_time")
        ).alias("arrival_diff")
    )
)

print(
    validation.select([
        "timestamp",
        "stop_id",
        "arrival_time",
        "arrival_diff"
    ])
)

shape: (1_122, 4)
┌────────────┬─────────┬──────────────┬──────────────┐
│ timestamp  ┆ stop_id ┆ arrival_time ┆ arrival_diff │
│ ---        ┆ ---     ┆ ---          ┆ ---          │
│ u64        ┆ str     ┆ i64          ┆ f64          │
╞════════════╪═════════╪══════════════╪══════════════╡
│ 1775433585 ┆ 400614  ┆ 1775433645   ┆ -60.0        │
│ 1775433585 ┆ 400614  ┆ 1775433665   ┆ -80.0        │
│ 1775433585 ┆ 400614  ┆ 1775433646   ┆ -61.0        │
│ 1775433613 ┆ 400614  ┆ 1775433645   ┆ -32.0        │
│ 1775433613 ┆ 400614  ┆ 1775433665   ┆ -52.0        │
│ …          ┆ …       ┆ …            ┆ …            │
│ 1775435357 ┆ 400636  ┆ null         ┆ null         │
│ 1775435384 ┆ 400636  ┆ null         ┆ null         │
│ 1775435411 ┆ 400636  ┆ null         ┆ null         │
│ 1775435439 ┆ 400636  ┆ null         ┆ null         │
│ 1775435467 ┆ 400636  ┆ null         ┆ null         │
└────────────┴─────────┴──────────────┴──────────────┘


In [ ]:
# %%
import polars as pl
import re
from pathlib import Path

pl.Config.set_tbl_cols(-1)

RT = Path("raw")

trip_updates = pl.read_parquet(RT / "trip_updates/2026-07-01.parquet")
vehicle_positions = pl.read_parquet(RT / "vehicle_positions/2026-07-01.parquet")

routes = pl.read_parquet("raw/processed_gtfs/routes.parquet")
trips = pl.read_parquet("raw/processed_gtfs/trips.parquet")

# %%
# ============================================================
# Normalize realtime trip_id -> static trip_id
# Strips MTA direction-prefixes like MV_/UP_/DW_/SB_
# ============================================================

def normalize_trip_id(tid: str) -> str:
    return re.sub(r"^[A-Z]{2}_", "", tid)

trip_updates = trip_updates.with_columns(
    pl.col("trip_id")
      .map_elements(normalize_trip_id, return_dtype=pl.String)
      .alias("trip_id_norm")
)

# %%
# Quick sanity check on match rate before going further
match_rate = (
    trip_updates
    .select("trip_id_norm")
    .unique()
    .join(trips.select("trip_id"), left_on="trip_id_norm", right_on="trip_id", how="inner")
    .height
    /
    trip_updates.select("trip_id_norm").unique().height
)
print(f"trip_id match rate: {match_rate:.2%}")

trip_id match rate: 0.00%
